# 007 — Lógica, algoritmos y complejidad computacional

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Lógica proposicional:** átomos y conectivos (¬, ∧, ∨, →). KB ⊨ α si α es verdadera en
todos los modelos de KB; verificar por tabla cuesta O(2ⁿ). La inferencia sintáctica
(*modus ponens*, resolución) evita enumerar modelos si es **correcta** y **completa**.
**SAT** (¿existe modelo?) es el problema NP-completo canónico (Cook, 1971).

**Complejidad:** O/Ω/Θ describen crecimiento asintótico. Jerarquía:
O(1) < O(log n) < O(n) < O(n log n) < O(n²) < O(2ⁿ) < O(n!). La frontera práctica es
polinomial vs. exponencial: duplicar hardware solo suma ~1 al n tratable de 2ⁿ.

- **P:** resoluble en tiempo polinomial. **NP:** *verificable* en polinomial.
- **NP-completo:** los más duros de NP (SAT, planificación); dureza del *peor caso*, no de
  toda instancia — los SAT solvers modernos resuelven instancias industriales enormes.
- **Indecidible:** el problema de la parada (Turing, 1936) no tiene algoritmo para todos
  los casos; por el teorema de Rice, ninguna propiedad semántica no trivial de programas
  lo tiene. Es el límite absoluto, por encima de "caro".

Tabla del muro exponencial (10⁹ asignaciones/s): n=20 → 1 ms · n=40 → 18 min ·
n=60 → 36 años · n=80 → 38 millones de años.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (1) llueve [hecho]; (2) llueve→suelo_mojado + (1) ⊢ suelo_mojado [MP];
(3) suelo_mojado→resbala + (2) ⊢ resbala [MP]. Sin el hecho `llueve`, la implicación
llueve→resbala **sí** es consecuencia: en todo modelo donde las dos reglas valen, si
llueve=V entonces resbala=V (encadenando), y si llueve=F la implicación es trivialmente
verdadera.

**Ejercicio 2.** 2⁴ = 16 modelos; 2³⁰ ≈ 1.07·10⁹ modelos ≈ 1.07 s a 10⁹/s. Con 60 átomos ya
serían ~36 años: la inferencia sintáctica llega en pocos pasos donde la enumeración es
inviable.

**Ejercicio 3.** log n < n < n log n < n³ < 2ⁿ < n!. Para n=30: log₂30 ≈ 5; 30; ~147;
27 000; ~1.07·10⁹; ~2.65·10³². Las dos últimas columnas son otro mundo: esa es la frontera
polinomial/exponencial.

**Ejercicio 4.** Si el motor es correcto, no: la corrección garantiza que solo se derivan
consecuencias lógicas (puede faltarle alguna si no es completo, pero no puede inventar).
El campo `evidence` debe listar hechos inspeccionables de la derivación.

In [ ]:
result = run_lab("logic", seed=7)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — encadenamiento hacia adelante mínimo
def forward_chain(reglas, hechos):
    hechos = set(hechos)
    cambio = True
    while cambio:
        cambio = False
        for premisa, conclusion in reglas:
            if premisa in hechos and conclusion not in hechos:
                hechos.add(conclusion)
                cambio = True
    return hechos

reglas = [("llueve", "suelo_mojado"), ("suelo_mojado", "resbala")]
print(forward_chain(reglas, ["llueve"]))  # incluye 'resbala'
assert "resbala" in forward_chain(reglas, ["llueve"])

In [ ]:
# Ejercicios 2-3 — el muro exponencial en números
import math
print("modelos 4 átomos:", 2**4)
print("modelos 30 átomos:", 2**30, "→", 2**30 / 1e9, "s a 1e9/s")
print("modelos 60 átomos:", 2**60 / 1e9 / 3600 / 24 / 365, "años")
n = 30
for nombre, val in [("log n", math.log2(n)), ("n", n), ("n log n", n*math.log2(n)),
                    ("n^3", n**3), ("2^n", 2**n), ("n!", math.factorial(n))]:
    print(f"{nombre:>8}: {val:.3g}")

## Reflexión

1. El laboratorio `logic` deriva conclusiones desde una base de conocimiento. ¿Su método es
   sintáctico (aplicar reglas) o semántico (enumerar modelos)? ¿Qué implica eso para su
   escalabilidad según la tabla del muro exponencial?
2. ¿Por qué "SAT es NP-completo" y "los SAT solvers resuelven instancias con millones de
   variables" no se contradicen? ¿Qué explota el solver que el análisis de peor caso ignora?
3. Da un ejemplo de tarea que un usuario podría pedirle a una IA y que sea indecidible en
   su forma general. ¿Qué versión relajada sí sería computable?